# Normal climatológica (ERA5-Land 1991–2020) e novas imagens (iNaturalist)

Rodem as células **em ordem**. As partes A e B são independentes: dá para rodar só uma delas.

**Antes de começar (só uma vez):**
1. Crie/entre na conta em https://cds.climate.copernicus.eu, abra a página do dataset *ERA5-Land monthly averaged data* e **aceite a licença** (aba *Download*, fim da página).
2. Copie a sua chave (perfil > *API Token*).
3. No Colab, clique no ícone de **chave** (Segredos) na barra lateral esquerda > *Adicionar segredo* > nome `CDS_KEY`, valor = sua chave > ative o acesso do notebook.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PASTA = '/content/drive/MyDrive/artigo_acai'      # tudo fica salvo aqui
os.makedirs(PASTA, exist_ok=True); os.chdir(PASTA); print(os.getcwd())

## A. Normal climatológica 1991–2020 e anomalias de 2024

In [ ]:
%pip install -q "cdsapi>=0.7" xarray netcdf4 pandas
from google.colab import userdata
open(os.path.expanduser('~/.cdsapirc'),'w').write('url: https://cds.climate.copernicus.eu/api\nkey: ' + userdata.get('CDS_KEY') + '\n')
print('chave configurada')

In [ ]:
%%writefile era5land_climatologia.py
"""
ERA5-Land: normal climatológica 1991–2020 (padrão OMM), anomalias de 2024 e estações DJF corretas
(item 5 do parecer técnico).

Pré-requisitos (rodar no Colab ou no computador de vocês):
  pip install "cdsapi>=0.7" xarray netcdf4 pandas
  Criar conta em https://cds.climate.copernicus.eu, aceitar a licença do ERA5-Land
  e salvar a chave em ~/.cdsapirc  (url: https://cds.climate.copernicus.eu/api  key: <sua-chave>)

Saídas (pasta era5_out/):
  climatologia_mensal.csv   normal 1991–2020: média, DP entre anos, mínimo e máximo espacial por mês
  anomalias_2024.csv        2024 menos a climatologia, por mês e por estação
  estacoes.csv              DJF 2023–24, MAM, JJA, SON 2024 (e DJF 2024–25 se baixado 2025)

Notas metodológicas para o texto:
  * Precipitação: no produto mensal do ERA5-Land, 'tp' é a acumulação DIÁRIA MÉDIA do mês, em metros.
    Portanto mm/dia = tp * 1000. (Multiplicar pelo nº de dias do mês dá o total mensal em mm.)
  * Temperatura/umidade/vento são calculados de duas formas: média de todas as horas (caracterização
    climática geral) e média das 18 UTC = 15 h local (estresse térmico vespertino). Use as duas no artigo.
  * Umidade relativa (Magnus): UR = 100 * exp(17.625*Td/(243.04+Td)) / exp(17.625*T/(243.04+T)), T e Td em °C.
  * Comparar a normal ERA5-Land com as Normais Climatológicas 1991–2020 do INMET (estação Belém e outras da região).
  * Validar contra estações INMET (ex.: Belém A201, Cametá A209/ outras disponíveis no BDMEP):
    extrair a célula ERA5-Land mais próxima e calcular viés, RMSE e correlação mensais.
"""
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

AREA = [-1.4, -49.8, -2.6, -48.5]          # N, W, S, E  (domínio do artigo)
NORMAL = (1991, 2020)                         # normal climatológica padrão da OMM (30 anos)
YEARS = [str(y) for y in range(1990, 2025)]  # 1990 (dez/1990 p/ DJF 1990–91) até 2024; acrescente "2025" para DJF 2024–25
OUT = Path("era5_out"); OUT.mkdir(exist_ok=True)
VARS = ["2m_temperature", "2m_dewpoint_temperature", "10m_u_component_of_wind",
        "10m_v_component_of_wind", "total_precipitation"]


def download():
    import cdsapi
    c = cdsapi.Client()
    common = dict(variable=VARS, year=YEARS, month=[f"{m:02d}" for m in range(1, 13)],
                  area=AREA, data_format="netcdf", download_format="unarchived")
    if not (OUT / "monthly_allhours.nc").exists():
        c.retrieve("reanalysis-era5-land-monthly-means",
                   dict(product_type=["monthly_averaged_reanalysis"], time=["00:00"], **common),
                   str(OUT / "monthly_allhours.nc"))
    if not (OUT / "monthly_18utc.nc").exists():
        c.retrieve("reanalysis-era5-land-monthly-means",
                   dict(product_type=["monthly_averaged_reanalysis_by_hour_of_day"], time=["18:00"],
                        **{k: v for k, v in common.items() if k != "variable"},
                        variable=VARS[:4]),
                   str(OUT / "monthly_18utc.nc"))


def derive(ds):
    tdim = "valid_time" if "valid_time" in ds.dims else "time"
    ds = ds.rename({tdim: "time"})
    out = xr.Dataset()
    T = ds["t2m"] - 273.15
    Td = ds["d2m"] - 273.15
    out["temp_C"] = T
    out["rh_pct"] = 100 * np.exp(17.625 * Td / (243.04 + Td)) / np.exp(17.625 * T / (243.04 + T))
    out["wind_ms"] = np.hypot(ds["u10"], ds["v10"])
    if "tp" in ds:
        out["precip_mm_day"] = ds["tp"] * 1000.0
    return out.squeeze(drop=True)


def spatial_stats(da):
    dims = [d for d in da.dims if d != "time"]
    return pd.DataFrame({"mean": da.mean(dims).values, "sp_min": da.min(dims).values,
                         "sp_max": da.max(dims).values, "sp_sd": da.std(dims).values},
                        index=pd.to_datetime(da["time"].values))


def season_label(ts):
    m, y = ts.month, ts.year
    if m == 12:
        return f"DJF {y}-{str(y + 1)[-2:]}"
    if m in (1, 2):
        return f"DJF {y - 1}-{str(y)[-2:]}"
    return {3: "MAM", 4: "MAM", 5: "MAM", 6: "JJA", 7: "JJA", 8: "JJA"}.get(m, "SON") + f" {y}"


def main():
    download()
    allh = derive(xr.open_dataset(OUT / "monthly_allhours.nc"))
    h18 = derive(xr.open_dataset(OUT / "monthly_18utc.nc")).rename(
        {"temp_C": "temp_C_15h", "rh_pct": "rh_pct_15h", "wind_ms": "wind_ms_15h"})
    frames = []
    for ds in (allh, h18):
        for v in ds.data_vars:
            s = spatial_stats(ds[v]); s["var"] = v; frames.append(s)
    df = pd.concat(frames).rename_axis("time").reset_index()
    df["year"], df["month"] = df.time.dt.year, df.time.dt.month

    base = df[(df.year >= NORMAL[0]) & (df.year <= NORMAL[1])]
    clim = base.groupby(["var", "month"]).agg(clim_mean=("mean", "mean"), interannual_sd=("mean", "std"),
                                              sp_min=("sp_min", "min"), sp_max=("sp_max", "max")).reset_index()
    clim.to_csv(OUT / "climatologia_mensal.csv", index=False)

    y24 = df[df.year == 2024].merge(clim, on=["var", "month"])
    y24["anomaly"] = y24["mean"] - y24["clim_mean"]
    y24["z"] = y24["anomaly"] / y24["interannual_sd"]
    y24.to_csv(OUT / "anomalias_2024.csv", index=False)

    df["season"] = df.time.apply(season_label)
    keep = ["DJF 2023-24", "MAM 2024", "JJA 2024", "SON 2024", "DJF 2024-25"]
    seas = (df[df.season.isin(keep)].groupby(["season", "var"])
            .agg(n_months=("mean", "size"), mean=("mean", "mean"),
                 sp_min=("sp_min", "min"), sp_max=("sp_max", "max")).reset_index())
    seas = seas[seas.n_months == 3]   # descarta estação incompleta
    seas.to_csv(OUT / "estacoes.csv", index=False)
    print(seas.pivot(index="season", columns="var", values="mean").round(2))


if __name__ == "__main__":
    main()


In [ ]:
# Pode levar de alguns minutos a algumas horas (fila do Copernicus). Se a sessão cair, rode de novo:
# os arquivos já baixados ficam em era5_out/ e não são baixados outra vez.
!python era5land_climatologia.py
import pandas as pd
display(pd.read_csv('era5_out/estacoes.csv'))

**Enviem para o Claude** os 3 arquivos da pasta `artigo_acai/era5_out/` no Drive: `climatologia_mensal.csv`, `anomalias_2024.csv`, `estacoes.csv`.

## B. Imagens do iNaturalist (licença aberta, com data e coordenadas)
Edite o e-mail em `HEADERS` dentro do script se quiser. O download leva ~30–60 min (a API limita 1 pedido/s).

In [ ]:
%%writefile baixar_imagens_inaturalist.py
"""
Baixa fotos com licença aberta e georreferenciadas do iNaturalist para ampliar o banco do YOLO.
Rodar no Google Colab (ou no computador):   python baixar_imagens_inaturalist.py

Por que iNaturalist: cada foto tem autor, licença, data e coordenadas (resolve a crítica de que as
imagens "não têm data nem localização") e o grau "research" significa identificação confirmada pela
comunidade. Usamos apenas licenças que permitem reuso com atribuição (CC0, CC-BY, CC-BY-NC).

Saída: imagens_inat/<grupo>/*.jpg  +  imagens_inat/metadados.csv  (autor, licença, data, lat, lon, URL)
-> O metadados.csv deve ir para o material suplementar / repositório (créditos das imagens).
-> Depois, subir as imagens para o Roboflow e anotar com polígonos (classe "suspected triatomine" ou "açaí").
"""
import csv
import time
from pathlib import Path

import requests

API = "https://api.inaturalist.org/v1"
LICENSES = "cc0,cc-by,cc-by-nc"
OUT = Path("imagens_inat")
HEADERS = {"User-Agent": "acai-yolo-research (contato: inserir e-mail do grupo)"}

GRUPOS = {
    # nome da pasta : (táxon, lugar, máximo de observações)
    "triatomineos_brasil": ("Triatominae", "Brazil", 600),
    "acai_euterpe_oleracea": ("Euterpe oleracea", "Brazil", 600),
    # imagens de "fundo" (sem barbeiro) ajudam a reduzir falsos positivos: insetos parecidos
    "parecidos_reduviidae_brasil": ("Reduviidae", "Brazil", 150),
    "parecidos_coreidae_brasil": ("Coreidae", "Brazil", 150),
}


def get(url, **params):
    for tentativa in range(5):
        r = requests.get(url, params=params, headers=HEADERS, timeout=60)
        if r.status_code == 429:       # limite de requisições: espera e tenta de novo
            time.sleep(10 * (tentativa + 1)); continue
        r.raise_for_status()
        time.sleep(1.0)                # respeita o limite de ~1 requisição/s da API
        return r.json()
    raise RuntimeError("API indisponível")


def taxon_id(nome):
    res = get(f"{API}/taxa", q=nome, per_page=5)["results"]
    exato = [t for t in res if t["name"].lower() == nome.lower()]
    return (exato or res)[0]["id"]


def place_id(nome):
    res = get(f"{API}/places/autocomplete", q=nome)["results"]
    return res[0]["id"]


def main():
    OUT.mkdir(exist_ok=True)
    meta_path = OUT / "metadados.csv"
    novo = not meta_path.exists()
    with open(meta_path, "a", newline="", encoding="utf-8") as fmeta:
        w = csv.writer(fmeta)
        if novo:
            w.writerow(["arquivo", "grupo", "observacao_id", "especie", "data", "lat", "lon",
                        "licenca", "autor", "atribuicao", "url_observacao", "url_foto"])
        for pasta, (taxon, lugar, maximo) in GRUPOS.items():
            tid, pid = taxon_id(taxon), place_id(lugar)
            (OUT / pasta).mkdir(exist_ok=True)
            n, page = 0, 1
            while n < maximo:
                data = get(f"{API}/observations", taxon_id=tid, place_id=pid, quality_grade="research",
                           photos="true", photo_license=LICENSES, per_page=100, page=page,
                           order_by="observed_on")
                if not data["results"]:
                    break
                for obs in data["results"]:
                    # 1 foto por observação: evita várias fotos do mesmo inseto (o que causaria vazamento entre treino e teste)
                    foto = next((p for p in obs.get("photos", []) if p.get("license_code") in LICENSES.split(",")), None)
                    if not foto:
                        continue
                    url = foto["url"].replace("square", "large")
                    nome = f"inat_{obs['id']}_{foto['id']}.jpg"
                    destino = OUT / pasta / nome
                    if not destino.exists():
                        try:
                            img = requests.get(url, headers=HEADERS, timeout=60); img.raise_for_status()
                            destino.write_bytes(img.content); time.sleep(0.5)
                        except Exception as e:
                            print("falhou", url, e); continue
                    lat, lon = (obs.get("location") or ",").split(",")
                    w.writerow([f"{pasta}/{nome}", pasta, obs["id"], (obs.get("taxon") or {}).get("name"),
                                obs.get("observed_on"), lat, lon, foto.get("license_code"),
                                (obs.get("user") or {}).get("login"), foto.get("attribution"),
                                obs.get("uri"), url])
                    n += 1
                    if n >= maximo:
                        break
                page += 1
            print(f"{pasta}: {n} imagens")


if __name__ == "__main__":
    main()


In [ ]:
!python baixar_imagens_inaturalist.py
import pandas as pd
m = pd.read_csv('imagens_inat/metadados.csv'); print(m.groupby('grupo').size()); m.head()

In [ ]:
# Galeria rápida para revisar: anotem os nomes dos arquivos a DESCARTAR (borradas, coleção/alfinete se não quiserem, inseto errado, etc.)
from PIL import Image; import matplotlib.pyplot as plt, glob
GRUPO = 'triatomineos_brasil'   # troque para 'acai_euterpe_oleracea' etc.
fs = sorted(glob.glob(f'imagens_inat/{GRUPO}/*.jpg'))
INICIO = 0                      # avance de 40 em 40
fig, ax = plt.subplots(5, 8, figsize=(24, 15))
for a, f in zip(ax.flat, fs[INICIO:INICIO+40]):
    a.imshow(Image.open(f)); a.set_title(os.path.basename(f)[5:20], fontsize=8); a.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Liste aqui os arquivos a descartar e rode a célula: eles vão para imagens_inat/_descartadas
DESCARTAR = []   # ex.: ['inat_12345_678.jpg', ...]
import shutil
os.makedirs('imagens_inat/_descartadas', exist_ok=True)
for f in DESCARTAR:
    for p in glob.glob(f'imagens_inat/*/{f}'): shutil.move(p, 'imagens_inat/_descartadas/')
print('ok')

### Depois do notebook (manual)
1. **Roboflow:** suba as pastas aprovadas no mesmo projeto, anote com polígonos (classes `a-ai` e `barbeiro`; as imagens de insetos parecidos ficam **sem anotação**, como fundo).
2. **Entomologista:** peça para conferir as anotações de barbeiro.
3. Gere uma nova versão **sem augmentation** e rode o notebook `YOLO11_acai_triatomineos.ipynb`.
4. Guarde o `metadados.csv`: ele vira a tabela de créditos das imagens no material suplementar.